<a href="https://colab.research.google.com/github/arinjain373/Major-Project/blob/main/tf_bert_final_layer_classify__.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel, BertTokenizer
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import os
from datetime import datetime
warnings.filterwarnings("ignore")

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)



In [ ]:
# Create directory for saving results
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
results_dir = f"results_{timestamp}"
os.makedirs(results_dir, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")



In [ ]:
# Load features and labels
visual_features = np.load("/kaggle/input/final-cmu-pro/visual_features.npy")  # Shape: (2199, 128)
audio_features = np.load("/kaggle/input/final-cmu-pro/audio_features.npy")    # Shape: (2199, 74)
labels_df = pd.read_csv("/kaggle/input/final-cmu-pro/labels.csv")             # Shape: (2199, 11)



In [ ]:
def min_max_scale(features, range_min=0, range_max=1):
    feat_min = np.min(features, axis=0)
    feat_max = np.max(features, axis=0)
    scaled = (features - feat_min) / (feat_max - feat_min + 1e-8)
    scaled = scaled * (range_max - range_min) + range_min  # Rescale to desired range
    return scaled

audio_features = min_max_scale(audio_features, -1, 1)  # Example: Scale to [-1, 1]
visual_features = min_max_scale(visual_features)



In [ ]:
# Split data into train/test/valid
def split_data(mode):
    idx = labels_df[labels_df["mode"] == mode].index
    return {
        "text": labels_df.loc[idx, "text"].values,
        "visual": visual_features[idx],
        "audio": audio_features[idx],
        "label": labels_df.loc[idx, "label"].values,  # Original sentiment score (-3 to +3)
        "annotation": labels_df.loc[idx, "annotation"].map({"Positive": 2, "Neutral": 1, "Negative": 0}).values,  # 3-class
        # Add binary classification: negative (0) and non-negative (1)
        "binary": (labels_df.loc[idx, "label"].values >= 0).astype(int)  # 0 for negative, 1 for non-negative
    }

train_data = split_data("train")
test_data = split_data("test")
valid_data = split_data("valid")



In [ ]:
# BERT Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

class MultimodalDataset(Dataset):
    def __init__(self, data):
        self.text = data["text"]
        self.visual = torch.FloatTensor(data["visual"])
        self.audio = torch.FloatTensor(data["audio"])
        self.annotation = torch.LongTensor(data["annotation"])
        self.binary = torch.LongTensor(data["binary"])

    def __len__(self):
        return len(self.text)

    def __getitem__(self, idx):
        text_encoded = tokenizer(
            self.text[idx],
            padding="max_length",
            truncation=True,
            max_length=50,
            return_tensors="pt"
        )
        return {
            "input_ids": text_encoded["input_ids"].squeeze(0),
            "attention_mask": text_encoded["attention_mask"].squeeze(0),
            "visual": self.visual[idx],
            "audio": self.audio[idx],
            "annotation": self.annotation[idx],
            "binary": self.binary[idx]
        }

# Create dataloaders
batch_size = 32
train_loader = DataLoader(MultimodalDataset(train_data), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(MultimodalDataset(test_data), batch_size=batch_size)
valid_loader = DataLoader(MultimodalDataset(valid_data), batch_size=batch_size)



In [ ]:
class TCT(nn.Module):
    def __init__(self, text_dim=768, audio_dim=74, visual_dim=128, num_heads=2):
        super().__init__()
        self.text_dim = text_dim
        self.audio_proj = nn.Linear(audio_dim, text_dim)
        self.visual_proj = nn.Linear(visual_dim, text_dim)

        self.multihead_attn = nn.MultiheadAttention(text_dim, num_heads)
        self.layer_norm = nn.LayerNorm(text_dim)
        self.ffn = nn.Sequential(
            nn.Linear(text_dim, 4 * text_dim),
            nn.ReLU(),
            nn.Linear(4 * text_dim, text_dim)
        )

    def forward(self, text, audio, visual):
        audio_proj = self.audio_proj(audio).unsqueeze(1)  # [batch, 1, 768]
        visual_proj = self.visual_proj(visual).unsqueeze(1)  # [batch, 1, 768]

        T_a = torch.matmul(text, audio_proj.transpose(1, 2))
        T_v = torch.matmul(text, visual_proj.transpose(1, 2))
        T_av = (T_a + T_v) / 2

        Q = text.transpose(0, 1)
        K = V = (audio_proj + visual_proj).transpose(0, 1)
        attn_output, _ = self.multihead_attn(Q, K, V)
        output = self.layer_norm(text + attn_output.transpose(0, 1))
        output = self.layer_norm(output + self.ffn(output))
        return output



In [ ]:
class TF_BERT(nn.Module):
    """TF-BERT Model for Classification Only"""
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.tct = TCT(text_dim=768, audio_dim=74, visual_dim=128)
        self.classifier = nn.Linear(768, 3)  # For 3-class classification
        self.binary_classifier = nn.Linear(768, 2)  # For binary classification

    def forward(self, input_ids, attention_mask, visual, audio):
        # BERT text encoding
        text_output = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

        # TCT fusion
        fused_output = self.tct(text_output, audio, visual)

        # Pooling for sentence-level representation
        pooled_output = fused_output.mean(dim=1)

        # Output heads
        logits = self.classifier(pooled_output)  # 3-class classification
        binary_logits = self.binary_classifier(pooled_output)  # Binary classification

        return logits, binary_logits

model = TF_BERT().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)



In [ ]:
def train_epoch(model, dataloader):
    model.train()
    total_loss = 0
    progress_bar = tqdm(dataloader, desc="Training")
    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        visual = batch["visual"].to(device)
        audio = batch["audio"].to(device)
        annotation = batch["annotation"].to(device)
        binary = batch["binary"].to(device)

        logits, binary_logits = model(input_ids, attention_mask, visual, audio)
        loss_cls = criterion(logits, annotation)  # 3-class loss
        loss_bin = criterion(binary_logits, binary)  # Binary loss
        loss = loss_cls + loss_bin

        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
    return total_loss / len(dataloader)



In [ ]:
def evaluate(model, dataloader):
    model.eval()
    preds, binary_preds = [], []
    anns, bins = [], []
    total_loss = 0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluation"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            visual = batch["visual"].to(device)
            audio = batch["audio"].to(device)
            annotation = batch["annotation"].to(device)
            binary = batch["binary"].to(device)

            logits, binary_logits = model(input_ids, attention_mask, visual, audio)
            loss_cls = criterion(logits, annotation)
            loss_bin = criterion(binary_logits, binary)
            loss = loss_cls + loss_bin
            total_loss += loss.item()

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            binary_preds.extend(torch.argmax(binary_logits, dim=1).cpu().numpy())
            anns.extend(annotation.cpu().numpy())
            bins.extend(binary.cpu().numpy())

    return np.array(preds), np.array(binary_preds), np.array(anns), np.array(bins), total_loss / len(dataloader)



In [ ]:
def compute_metrics(preds, binary_preds, anns, bins):
    # 3-class classification metrics
    acc = accuracy_score(anns, preds)
    f1_3class = f1_score(anns, preds, average="weighted")

    # Binary classification metrics (acc2)
    acc2 = accuracy_score(bins, binary_preds)
    f1_binary = f1_score(bins, binary_preds, average="binary")

    return {
        "ACC-3": acc,
        "F1-3class": f1_3class,
        "ACC-2": acc2,
        "F1-binary": f1_binary
    }

def plot_confusion_matrix(anns, preds, title="Confusion Matrix", save_path=None):
    cm = confusion_matrix(anns, preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Neutral", "Positive"])
    plt.figure(figsize=(8, 6))
    disp.plot(cmap='Blues')
    plt.title(title)
    if save_path:
        plt.savefig(save_path)
    plt.close()

def plot_binary_confusion_matrix(bins, binary_preds, title="Binary Confusion Matrix", save_path=None):
    cm = confusion_matrix(bins, binary_preds)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Negative", "Non-negative"])
    plt.figure(figsize=(8, 6))
    disp.plot(cmap='Blues')
    plt.title(title)
    if save_path:
        plt.savefig(save_path)
    plt.close()

def plot_loss_curves(train_losses, valid_losses, save_path=None):
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label="Training Loss")
    plt.plot(valid_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss Curves")
    plt.legend()
    if save_path:
        plt.savefig(save_path)
    plt.close()



In [ ]:
# Training loop with early stopping
train_losses = []
valid_losses = []
best_valid_loss = float('inf')
patience = 5
no_improvement = 0

max_epochs = 50
for epoch in range(max_epochs):
    # Training
    epoch_train_loss = train_epoch(model, train_loader)
    train_losses.append(epoch_train_loss)

    # Validation
    preds, binary_preds, anns, bins, epoch_valid_loss = evaluate(model, valid_loader)
    valid_losses.append(epoch_valid_loss)
    metrics = compute_metrics(preds, binary_preds, anns, bins)

    print(f"Epoch {epoch+1}/{max_epochs}")
    print(f"Train Loss: {epoch_train_loss:.4f}, Valid Loss: {epoch_valid_loss:.4f}")
    print(f"Validation Metrics: {metrics}")

    # Early stopping check
    if epoch_valid_loss < best_valid_loss:
        best_valid_loss = epoch_valid_loss
        no_improvement = 0
        # Save best model
        torch.save(model.state_dict(), os.path.join(results_dir, "best_model.pt"))
    else:
        no_improvement += 1
        if no_improvement >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs!")
            break


model.load_state_dict(torch.load(os.path.join(results_dir, "best_model.pt")))



Evaluation: 100%|██████████| 8/8 [00:00<00:00, 18.06it/s]


Epoch 1/50
Train Loss: 1.4317, Valid Loss: 1.2730
Validation Metrics: {'ACC-3': 0.7379912663755459, 'F1-3class': 0.7152084138817665, 'ACC-2': 0.6943231441048034, 'F1-binary': 0.7784810126582278}


Evaluation: 100%|██████████| 8/8 [00:00<00:00, 18.27it/s]


Epoch 2/50
Train Loss: 1.0018, Valid Loss: 0.9086
Validation Metrics: {'ACC-3': 0.7991266375545851, 'F1-3class': 0.7772516536790345, 'ACC-2': 0.8165938864628821, 'F1-binary': 0.8372093023255814}


Evaluation: 100%|██████████| 8/8 [00:00<00:00, 18.08it/s]


...

Evaluation: 100%|██████████| 8/8 [00:00<00:00, 18.24it/s]


Epoch 6/50
Train Loss: 0.1797, Valid Loss: 1.3915
Validation Metrics: {'ACC-3': 0.7641921397379913, 'F1-3class': 0.7482684976546424, 'ACC-2': 0.7947598253275109, 'F1-binary': 0.8212927756653992}


Evaluation: 100%|██████████| 8/8 [00:00<00:00, 18.20it/s]


Epoch 7/50
Train Loss: 0.1341, Valid Loss: 1.5555
Validation Metrics: {'ACC-3': 0.7641921397379913, 'F1-3class': 0.756544657633525, 'ACC-2': 0.7991266375545851, 'F1-binary': 0.8257575757575758}
Early stopping triggered after 7 epochs!


<All keys matched successfully>

In [ ]:
# Test evaluation
preds, binary_preds, anns, bins, test_loss = evaluate(model, test_loader)
metrics = compute_metrics(preds, binary_preds, anns, bins)
print(f"Test Metrics: {metrics}")
print(f"Test ACC-2: {metrics['ACC-2']:.4f}")
print(f"Test F1-binary: {metrics['F1-binary']:.4f}")

# Plot and save loss curves
plot_loss_curves(train_losses, valid_losses,
                save_path=os.path.join(results_dir, "loss_curves.png"))

# Plot and save confusion matrices
plot_confusion_matrix(anns, preds, title="Test Confusion Matrix (3-class)",
                    save_path=os.path.join(results_dir, "confusion_matrix_3class.png"))

plot_binary_confusion_matrix(bins, binary_preds, title="Test Binary Confusion Matrix",
                           save_path=os.path.join(results_dir, "confusion_matrix_binary.png"))

# Save metrics to file
with open(os.path.join(results_dir, "metrics.txt"), "w") as f:
    f.write("Test Metrics:\n")
    for k, v in metrics.items():
        f.write(f"{k}: {v:.4f}\n")

Evaluation: 100%|██████████| 22/22 [00:01<00:00, 16.55it/s]


Test Metrics: {'ACC-3': 0.8017492711370262, 'F1-3class': 0.7833942492138172, 'ACC-2': 0.7959183673469388, 'F1-binary': 0.7643097643097644}
Test ACC-2: 0.7959
Test F1-binary: 0.7643


<Figure size 800x600 with 0 Axes>

<Figure size 800x600 with 0 Axes>